# Session 4 — Feature Engineering

**Data Science & Machine Learning: from Skills to the Project**  
**Duration:** 90 minutes  
**Case study:** transform the book catalogue into reproducible ML-ready features.

## Outcomes and agenda

You will design problem-led features; derive date and numerical attributes; choose categorical encodings; demonstrate interactions, polynomial features and leakage-safe target encoding; and build a reproducible `ColumnTransformer` pipeline.

| Minutes | Activity |
|---:|---|
| 0–10 | Feature design principles |
| 10–25 | Date and numerical features |
| 25–40 | Categorical encoding |
| 40–50 | Interactions and polynomial features |
| 50–60 | Target encoding and leakage |
| 60–75 | Pipeline and guided lab |
| 75–90 | Independent challenge and reflection |

The target is labelled clearly if it must be simulated. It is educational synthetic data—not evidence of real-world predictive performance.

## 1. Design features from the problem

A useful feature expresses information relevant to the prediction question, is available at prediction time, and can be calculated consistently for future records. Start from domain reasoning, not a long list of transformations. Avoid identifiers as predictors unless their meaning is justified; an ID can accidentally encode collection order or leak the answer. Check units, missingness, and whether the feature would exist when a real prediction is made.

The catalogue has no observed demand label. We therefore construct `high_demand` from catalogue attributes solely to demonstrate preprocessing. Never present results using this synthetic label as real evidence.

In [ ]:
from pathlib import Path
import re

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import KFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    OrdinalEncoder,
    PolynomialFeatures,
    StandardScaler,
)

PROCESSED_DIR = Path("data") / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
CLEAN_PATH = PROCESSED_DIR / "catalogue_clean.csv"
RNG_SEED = 1729
AS_OF_DATE = pd.Timestamp("2026-10-26")

def fallback_catalogue():
    """Build a small, deterministic book catalogue when Session 3 output is absent."""
    rows = [
        ("B001", "The Hobbit", "Fantasy", 8.99, 4.5, True),
        ("B002", "Pride and Prejudice", "Fiction", 7.50, 4.0, True),
        ("B003", "A Brief History of Time", "Science", 9.25, 4.2, False),
        ("B004", "The Left Hand of Darkness", "Science Fiction", 8.25, 4.7, True),
        ("B005", "Dune", "Science Fiction", 12.00, 4.8, True),
        ("B006", "Kindred", "Fiction", 11.50, 4.3, True),
        ("B007", "The Dispossessed", "Science Fiction", 10.25, 4.6, True),
        ("B008", "The Name of the Rose", "Mystery", 14.00, 4.1, True),
        ("B009", "Invisible Cities", "Travel", 9.75, 4.0, True),
        ("B010", "The Fifth Season", "Fantasy", 13.50, 4.7, True),
        ("B011", "The Martian", "Science Fiction", 16.00, 4.2, True),
        ("B012", "Piranesi", "Fantasy", 10.00, 4.4, True),
        ("B013", "The Overstory", "Fiction", 15.25, 4.0, False),
        ("B014", "An Immense World", "Nature", 18.00, 4.6, True),
        ("B015", "The Book of Why", "Science", 21.00, 4.0, False),
        ("B016", "North Woods", "Fiction", 17.50, 4.8, True),
    ]
    return pd.DataFrame(rows, columns=["record_id", "title", "category", "price_gbp", "rating", "available"])

if CLEAN_PATH.is_file():
    catalogue = pd.read_csv(CLEAN_PATH)
    input_source = str(CLEAN_PATH)
else:
    catalogue = fallback_catalogue()
    input_source = "deterministic standalone fallback"

for column, default in {"record_id": "unknown", "title": "Untitled", "category": "Uncategorised", "price_gbp": np.nan, "rating": np.nan, "available": pd.NA}.items():
    if column not in catalogue.columns:
        catalogue[column] = default
catalogue = catalogue.copy()
catalogue["title"] = catalogue["title"].fillna("Untitled").astype(str).str.strip()
catalogue["category"] = catalogue["category"].fillna("Uncategorised").astype(str).str.strip().str.title()
catalogue["price_gbp"] = pd.to_numeric(catalogue["price_gbp"], errors="coerce")
catalogue["rating"] = pd.to_numeric(catalogue["rating"], errors="coerce")
catalogue["available"] = catalogue["available"].map(
    lambda value: value if isinstance(value, (bool, np.bool_)) else str(value).strip().casefold() in {"true", "yes", "1"} if pd.notna(value) else False
).astype(bool)
catalogue = catalogue.loc[catalogue["record_id"].notna()].drop_duplicates("record_id").reset_index(drop=True)
if len(catalogue) < 8:
    raise ValueError("Feature engineering needs at least 8 catalogue rows for a train/test demonstration")

if "listed_on" not in catalogue.columns:
    catalogue["listed_on"] = pd.date_range("2025-01-01", periods=len(catalogue), freq="17D").astype(str)
catalogue["listed_on"] = pd.to_datetime(catalogue["listed_on"], errors="coerce")

# Synthetic educational label: plausible catalogue signals plus fixed, controlled noise.
if "high_demand" not in catalogue.columns:
    price_rank = catalogue["price_gbp"].rank(pct=True).fillna(0.5)
    rating_rank = catalogue["rating"].rank(pct=True).fillna(0.5)
    score = 0.55 * rating_rank + 0.30 * catalogue["available"].astype(float) + 0.15 * (1 - price_rank)
    labels = (score.rank(method="first") > len(score) / 2).to_numpy(copy=True)
    rng = np.random.default_rng(RNG_SEED)
    # Swap one positive and negative label to add controlled, class-balanced noise.
    positive_idx = rng.choice(np.flatnonzero(labels))
    negative_idx = rng.choice(np.flatnonzero(~labels))
    labels[[positive_idx, negative_idx]] = ~labels[[positive_idx, negative_idx]]
    catalogue["high_demand"] = labels
    target_description = "synthetic educational target; not real-world predictive evidence"
else:
    catalogue["high_demand"] = catalogue["high_demand"].astype(bool)
    target_description = "target supplied in input data"

print(f"Loaded {len(catalogue)} records from {input_source}")
print("Target:", target_description)
print("Target counts:", catalogue["high_demand"].value_counts().sort_index().to_dict())
assert catalogue["record_id"].is_unique
assert catalogue["high_demand"].nunique() == 2
assert catalogue["high_demand"].value_counts().min() >= 2

## 2. Date, numerical and safe-ratio features

Features should be computable at prediction time. From a listing date, derive age in days and month-cycle components. From a title, derive a simple length; from price and rating, derive price per rating point only when the denominator is positive. Protect ratios against zero or missing denominators. Date features use a fixed reference date for reproducibility; in a production task, define an as-of date that matches the prediction setting.

In [ ]:
features = catalogue.copy()
features["days_since_listed"] = (AS_OF_DATE - features["listed_on"]).dt.days.astype(float)
month = features["listed_on"].dt.month
features["listed_month_sin"] = np.sin(2 * np.pi * month / 12)
features["listed_month_cos"] = np.cos(2 * np.pi * month / 12)
features["title_length"] = features["title"].str.len().astype(float)
features["title_word_count"] = features["title"].str.split().str.len().astype(float)
features["price_per_rating"] = features["price_gbp"].div(features["rating"].where(features["rating"] > 0))
features["price_band"] = pd.cut(
    features["price_gbp"], bins=[-np.inf, 10, 20, np.inf], labels=["budget", "mid", "premium"], right=True
).astype("string").fillna("unknown")
assert features["days_since_listed"].notna().all()
assert features.loc[features["rating"].isna() | features["rating"].le(0), "price_per_rating"].isna().all()
assert features["title_word_count"].ge(1).all()
print(features[["title", "days_since_listed", "listed_month_sin", "title_word_count", "price_per_rating", "price_band"]].head(5).to_string(index=False))

## 3. Categorical features: one-hot or ordinal?

A category such as genre has no natural order: use one-hot encoding, not `LabelEncoder` or arbitrary integer codes. `OneHotEncoder(handle_unknown='ignore')` creates indicator columns and tolerates categories first seen at prediction time. Ordinal encoding is appropriate only when order is meaningful and explicitly defined (for example, low < medium < high). A high-cardinality field creates many columns, increases memory and overfitting risk, and may encode identifiers rather than useful signal.

In [ ]:
genre_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
genre_matrix = genre_encoder.fit_transform(features[["category"]])
genre_names = genre_encoder.get_feature_names_out(["category"])
priority_order = ["low", "medium", "high"]
priority_encoder = OrdinalEncoder(categories=[priority_order], handle_unknown="use_encoded_value", unknown_value=-1)
priority_example = priority_encoder.fit_transform(pd.DataFrame({"priority": ["low", "medium", "high"]}))
assert genre_matrix.shape[1] == len(genre_names)
assert priority_example.ravel().tolist() == [0.0, 1.0, 2.0]
unknown_genre = genre_encoder.transform(pd.DataFrame({"category": ["Unseen genre"]}))
assert unknown_genre.sum() == 0
print("One-hot genre columns:", len(genre_names), "| examples:", genre_names[:4].tolist())
print("Ordered priority mapping:", dict(zip(priority_order, priority_example.ravel().tolist())))
print("Unseen genre safely ignored:", unknown_genre.sum() == 0)

## 4. Interactions and polynomial features

An interaction represents a combined effect, such as price × rating. Polynomial expansion adds powers and pairwise products that a linear model could not otherwise express. It can help when domain reasoning supports nonlinear relationships, but the number of features grows rapidly: with *p* input columns and degree *d*, expansion can become large. Use only a small, selected set. This preview uses complete rows because polynomial terms themselves do not estimate data statistics; in a modelling workflow, place feature transformations inside the training-fitted pipeline.

In [ ]:
interaction_demo = features[["price_gbp", "rating"]].dropna().copy()
interaction_demo["price_x_rating"] = interaction_demo["price_gbp"] * interaction_demo["rating"]
poly = PolynomialFeatures(degree=2, include_bias=False)
poly_matrix = poly.fit_transform(interaction_demo[["price_gbp", "rating"]])
poly_names = poly.get_feature_names_out(["price_gbp", "rating"])
assert "price_gbp rating" in poly_names
print("Explicit interaction and polynomial terms:")
print(interaction_demo[["price_gbp", "rating", "price_x_rating"]].head(3).round(2).to_string(index=False))
print("Degree-2 terms:", poly_names.tolist(), "| columns:", poly_matrix.shape[1])

## 5. Split before learning: target leakage

Never include `high_demand` in predictors. Split records into training and test sets before fitting encoders or any other transformations that learn from the data. A target-derived feature is particularly risky: if test labels contribute to an encoding, evaluation leaks the answers. For the following target-encoding demonstration, training rows receive out-of-fold encodings; validation/test encodings use a mapping fitted on training data only.

In [ ]:
TARGET = "high_demand"
EXCLUDED = {"record_id", "title", "author", "listed_on", TARGET}
candidate_columns = [column for column in [
    "price_gbp", "rating", "available", "category", "price_band",
    "days_since_listed", "listed_month_sin", "listed_month_cos", "title_length",
    "title_word_count", "price_per_rating",
] if column in features.columns and column not in EXCLUDED]
X = features[candidate_columns].copy()
y = features[TARGET].astype(int).copy()
if y.value_counts().min() < 2:
    raise ValueError("Need at least two examples per target class for a stratified train/test split")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RNG_SEED, stratify=y
)
assert TARGET not in X_train.columns and TARGET not in X_test.columns
assert X_train.index.intersection(X_test.index).empty
assert set(y_train.unique()) == set(y_test.unique())
print("Predictor columns:", candidate_columns)
print("Split sizes:", len(X_train), "train /", len(X_test), "test")

## 6. Target encoding with smoothing and out-of-fold training values

Target encoding replaces a category with a smoothed estimate of its target rate. Smoothing shrinks small groups towards the training-set prior, reducing extreme estimates for rare categories. A naive encoding calculated on the same row's label leaks that label into its feature. Here each training row is encoded using a mapping learned from other folds; test rows use a mapping from all training rows. Test labels are never consulted.

In [ ]:
def smoothed_mapping(categories, target, smoothing=5.0):
    prior = float(target.mean())
    stats = pd.DataFrame({"category": categories.astype(str).to_numpy(), "target": target.to_numpy()}).groupby("category")["target"].agg(["mean", "count"])
    encoded = (stats["count"] * stats["mean"] + smoothing * prior) / (stats["count"] + smoothing)
    return prior, encoded.to_dict()

def cross_fitted_target_encoding(train_categories, train_target, valid_categories, n_splits=3, smoothing=5.0, seed=RNG_SEED):
    if train_categories.isna().any() or valid_categories.isna().any():
        raise ValueError("Fill or label missing categories before target encoding")
    if train_target.nunique() < 2 or train_target.value_counts().min() < n_splits:
        raise ValueError("Target classes need enough examples for the requested encoding folds")
    encoded_train = np.empty(len(train_target), dtype=float)
    splitter = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for fit_idx, heldout_idx in splitter.split(train_categories):
        prior, mapping = smoothed_mapping(train_categories.iloc[fit_idx], train_target.iloc[fit_idx], smoothing)
        encoded_train[heldout_idx] = train_categories.iloc[heldout_idx].astype(str).map(mapping).fillna(prior).to_numpy()
    prior, full_mapping = smoothed_mapping(train_categories, train_target, smoothing)
    encoded_valid = valid_categories.astype(str).map(full_mapping).fillna(prior).to_numpy()
    return encoded_train, encoded_valid, full_mapping, prior

category_oof, category_test, category_mapping, category_prior = cross_fitted_target_encoding(
    X_train["category"], y_train, X_test["category"]
)
assert np.isfinite(category_oof).all() and np.isfinite(category_test).all()
assert np.all((0 <= category_oof) & (category_oof <= 1))
assert np.all((0 <= category_test) & (category_test <= 1))
print("Training OOF values (first five):", np.round(category_oof[:5], 3).tolist())
print("Training-only prior:", round(category_prior, 3), "| mapped training categories:", len(category_mapping))

## 7. Guided lab — reproducible `ColumnTransformer` and `Pipeline`

Use median imputation and standard scaling for numerical columns; most-frequent imputation and one-hot encoding for categorical columns. The encoder ignores unknown categories. `ColumnTransformer` applies the right transformation to each column group, while `Pipeline` keeps the fitted transformations together. Fit only on `X_train`; transform test data without refitting.

In [ ]:
numeric_columns = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_columns = [column for column in X_train.columns if column not in numeric_columns]
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_columns),
    ("categorical", categorical_pipeline, categorical_columns),
], remainder="drop", verbose_feature_names_out=True)
preprocessing_pipeline = Pipeline([("preprocessor", preprocessor)])
train_features = preprocessing_pipeline.fit_transform(X_train)
test_features = preprocessing_pipeline.transform(X_test)
feature_names = preprocessing_pipeline.get_feature_names_out()
assert train_features.shape[1] == test_features.shape[1] == len(feature_names)
assert np.isfinite(train_features).all() and np.isfinite(test_features).all()
assert len(feature_names) == len(set(feature_names))
print("Feature matrix shapes:", train_features.shape, test_features.shape)
print("Feature names:", feature_names.tolist())

## 8. Guided lab — verify repeatability and inspect the output

Transform the same training input again and compare it with the original result. The check confirms that a fitted preprocessing pipeline is deterministic for fixed input and fitted state. Inspect a small feature-name preview; do not dump a large expanded matrix.

In [ ]:
train_features_repeat = preprocessing_pipeline.transform(X_train)
assert np.allclose(train_features, train_features_repeat)
assert list(X_train.columns) == list(X_test.columns)
assert preprocessing_pipeline.named_steps["preprocessor"].transformers_[0][2] == numeric_columns
preview = pd.DataFrame(train_features[:3], columns=feature_names)
print("Deterministic repeat transform:", np.allclose(train_features, train_features_repeat))
print("Aligned train/test input columns:", list(X_train.columns) == list(X_test.columns))
print("First transformed rows and up to six columns:\n", preview.iloc[:, :6].round(2).to_string(index=False))

## 9. Independent 20-minute challenge — add two features

Add two useful features to the catalogue using only fields available before demand is known. Suggestions: title word count, a safe price-per-rating ratio, a price band, or listing-month sine/cosine. State why each feature might help and how you handled missing or zero values. Verify that the engineered columns align with the original row index, contain no infinities, and do not include `high_demand` as an input.

In [ ]:
challenge_features = features.copy()
challenge_features["title_word_count_check"] = challenge_features["title"].fillna("").str.split().str.len().astype(float)
challenge_features["price_per_rating_check"] = challenge_features["price_gbp"].div(
    challenge_features["rating"].where(challenge_features["rating"] > 0)
)
assert challenge_features.index.equals(features.index)
assert np.isfinite(challenge_features["title_word_count_check"]).all()
assert not np.isinf(challenge_features["price_per_rating_check"].dropna()).any()
assert TARGET not in candidate_columns
assert challenge_features["title_word_count_check"].ge(0).all()
print(challenge_features[["title", "title_word_count_check", "price_per_rating_check"]].head(5).round(2).to_string(index=False))
print("Two additional features validated; target excluded from predictors.")

## 10. Export reproducible artefacts

Save (1) the engineered, still-readable feature table and target, (2) train/test raw feature partitions with labels, (3) the transformed ML-ready matrix with a split marker, and (4) the fitted preprocessing pipeline. The pipeline is fitted on training rows only. These small local artefacts are deterministic for the same input and seed; a real model workflow should persist its exact split and data provenance too.

In [ ]:
FEATURES_PATH = PROCESSED_DIR / "catalogue_features.csv"
TRAIN_PATH = PROCESSED_DIR / "catalogue_train.csv"
TEST_PATH = PROCESSED_DIR / "catalogue_test.csv"
TRANSFORMED_PATH = PROCESSED_DIR / "catalogue_ml_ready.csv"
PIPELINE_PATH = PROCESSED_DIR / "catalogue_preprocessing_pipeline.joblib"

export_columns = list(dict.fromkeys(candidate_columns + [TARGET]))
features[export_columns].to_csv(FEATURES_PATH, index=False, encoding="utf-8")
X_train.assign(high_demand=y_train).to_csv(TRAIN_PATH, index=False, encoding="utf-8")
X_test.assign(high_demand=y_test).to_csv(TEST_PATH, index=False, encoding="utf-8")
train_export = pd.DataFrame(train_features, columns=feature_names, index=X_train.index).assign(high_demand=y_train, split="train")
test_export = pd.DataFrame(test_features, columns=feature_names, index=X_test.index).assign(high_demand=y_test, split="test")
ml_ready = pd.concat([train_export, test_export]).sort_index()
ml_ready.to_csv(TRANSFORMED_PATH, index=False, encoding="utf-8")
joblib.dump(preprocessing_pipeline, PIPELINE_PATH)

reloaded = joblib.load(PIPELINE_PATH)
assert np.allclose(reloaded.transform(X_test), test_features)
assert ml_ready.drop(columns=["high_demand", "split"]).notna().all().all()
assert set(ml_ready["split"]) == {"train", "test"}
assert ml_ready["high_demand"].notna().all()
assert pd.read_csv(FEATURES_PATH).shape[0] == len(features)
print("Saved feature table:", FEATURES_PATH)
print("Saved split files:", TRAIN_PATH, "and", TEST_PATH)
print("Saved transformed matrix:", TRANSFORMED_PATH, ml_ready.shape)
print("Saved fitted preprocessing pipeline:", PIPELINE_PATH)
print("Reloaded pipeline reproduces test transform:", np.allclose(reloaded.transform(X_test), test_features))

## 11. Reflection, verification checklist and recap

1. Which engineered feature has the clearest connection to the prediction question, and what assumption does it make?
2. Why is one-hot encoding appropriate for genre but ordinal encoding inappropriate without a real order?
3. How does out-of-fold target encoding prevent a training row's label from influencing its own encoded value?
4. What are two risks of adding many categories or polynomial terms?
5. Why must imputation, scaling and category vocabularies be learned from training data only?

**Verification checklist:** target excluded from predictors; date features use a defined reference date; ratios guard invalid denominators; unknown categories are handled; train/test columns align; preprocessing fits training rows only; transformed values are finite and reproducible; pipeline reload reproduces its transform; synthetic target clearly labelled.

**Expected points:** features should be available and justified at prediction time; unordered genres need indicator columns; out-of-fold encodings learn category statistics from other folds only; feature explosion increases cost and overfitting risk; fitting on all data leaks test distribution or target information. Revisit the assumptions when replacing synthetic labels with observed business outcomes.

**Recap:** derive only defensible features; encode unordered categories without invented order; keep interactions and target encoding leakage-safe; split first; fit imputers, scalers and encoders through a training-only `ColumnTransformer`/`Pipeline`; save the fitted artefacts and split for reproducibility.